# Chatterbox TTS — Dharmendra voice clone (EN + Hindi) on Colab

Runs the chatterbox-tts server with Dharmendra voice clones (T4 GPU supported for ~2s generation, CPU supported as fallback).

**Setup:**
1. **Runtime → Change runtime type → T4 GPU** (Recommended: ~2s generation per sentence!)
2. **Runtime → Run all**

**Timings:** ~2 min setup on GPU (CPU takes ~15 min).

In [ ]:
# 1) Config + optional Google Drive model cache
USE_DRIVE_CACHE = True
DRIVE_CACHE_DIR = "/content/drive/MyDrive/chatterbox_tts_cache"
PORT = 8001

import os
os.makedirs("/content/hf_cache", exist_ok=True)

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(f"{DRIVE_CACHE_DIR}/hf_cache"):
        print("Restoring HF cache from Drive...")
        !cp -rn "{DRIVE_CACHE_DIR}/hf_cache/." /content/hf_cache/ 2>/dev/null
        n = sum(len(fs) for _, _, fs in os.walk("/content/hf_cache"))
        print(f"restored ({n} files)")
    else:
        print("No Drive cache yet — models will download on first voiceover.")


In [ ]:
# 2) Install chatterbox-tts (Smart GPU vs CPU setup with ABI matching)
import subprocess, sys

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print("FAILED:", cmd); print(r.stdout[-800:]); print(r.stderr[-800:])
    return r.returncode

has_gpu = False
try:
    import torch
    has_gpu = torch.cuda.is_available()
except Exception:
    pass

if not has_gpu:
    r = subprocess.run("nvidia-smi", shell=True, capture_output=True)
    if r.returncode == 0:
        has_gpu = True

if has_gpu:
    import torch
    cuda_ver = getattr(torch.version, "cuda", None) or "12.4"
    cu_tag = "cu" + cuda_ver.replace(".", "")
    print(f"🚀 GPU detected (CUDA {cuda_ver}). Syncing matching torchaudio ({cu_tag})...")
    sh(f"pip install -q --no-cache-dir torchaudio torchvision --index-url https://download.pytorch.org/whl/{cu_tag}")
    sh("pip install -q --no-cache-dir chatterbox-tts==0.1.7")
else:
    print("💻 CPU mode detected. Installing pinned CPU PyTorch...")
    sh("apt-get -qq install -y libgomp1 > /dev/null 2>&1")
    sh("pip install -q torch==2.6.0+cpu torchvision==0.21.0+cpu torchaudio==2.6.0+cpu --index-url https://download.pytorch.org/whl/cpu")
    sh("pip install -q chatterbox-tts==0.1.7")

import torch, torchaudio
print("PyTorch version:", torch.__version__)
print("torchaudio version:", torchaudio.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
print("installed")



In [ ]:
# 3) Get server + voice refs, wire Dharmendra clones, start server
import os, json, subprocess, time, urllib.request

os.chdir("/content")
if os.path.exists("/content/chatterbox-tts"):
    !rm -rf /content/chatterbox-tts
!git clone -q https://github.com/Dkpiec/chatterbox-tts.git

with open("/content/chatterbox-tts/current_clone.json", "w") as f:
    json.dump({
        "en":  {"audio_wav": "/content/chatterbox-tts/voice_ref.wav",    "exaggeration": 0.4},
        "mtl": {"audio_wav": "/content/chatterbox-tts/voice_ref_hi.wav", "exaggeration": 0.4},
    }, f, indent=1)

# pre-flight check
try:
    from chatterbox.tts import ChatterboxTTS  # noqa: F401
    print("chatterbox import OK")
except Exception as e:
    raise RuntimeError(f"PREFLIGHT FAILED — chatterbox cannot import: {e!r}") from e

env = dict(os.environ, HF_HOME="/content/hf_cache", PORT=str(PORT), IDLE_UNLOAD_S="3600")
log = open("/content/chatterbox_server.log", "w")
proc = subprocess.Popen(["python", "/content/chatterbox-tts/server.py"],
                        env=env, stdout=log, stderr=subprocess.STDOUT)
print("server pid:", proc.pid)

for _ in range(30):
    try:
        r = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
        print("health:", r.read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    print("server did not respond — check /content/chatterbox_server.log")
    !tail -20 /content/chatterbox_server.log


In [ ]:
# 4) Public HTTPS tunnel
import subprocess, time, re
!curl -sLo /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
tun_log = open("/content/tunnel.log", "w")
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                  "--no-autoupdate"], stdout=tun_log, stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"(https://[a-z0-9\-]+\.trycloudflare\.com)", open("/content/tunnel.log").read())
    if m:
        PUBLIC_URL = m.group(1)
        break
print("PUBLIC URL:", PUBLIC_URL)
if PUBLIC_URL:
    print("Health check:", PUBLIC_URL + "/health")


In [ ]:
# 5) Test English Voiceover
import json, urllib.request, traceback
from IPython.display import Audio, display

body = json.dumps({"text": "Hello, this is a test of the cloned voice. It should sound like Dharmendra.",
                   "lang": "en", "exaggeration": 0.4}).encode()
req = urllib.request.Request(f"http://127.0.0.1:{PORT}/tts", data=body,
                             headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(req, timeout=1800) as r:
        wav = r.read()
    open("/content/test_en.wav", "wb").write(wav)
    print(f"{len(wav)} bytes generated")
    display(Audio("/content/test_en.wav"))
except urllib.error.HTTPError as e:
    print(f"HTTP {e.code}: {e.reason}")
    !tail -40 /content/chatterbox_server.log
except Exception as ex:
    print(f"Error: {ex}")
    !tail -40 /content/chatterbox_server.log


In [ ]:
# 6) Test Hindi Voiceover
from google.colab import files

TEXT = "नमस्ते, यह धर्मेंद्र की आवाज़ का टेस्ट है।"
LANG = "hi"

body = json.dumps({"text": TEXT, "lang": LANG, "exaggeration": 0.4}).encode()
req = urllib.request.Request(f"http://127.0.0.1:{PORT}/tts", data=body,
                             headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(req, timeout=3600) as r:
        wav = r.read()
    out = f"/content/voiceover_{LANG}.wav"
    open(out, "wb").write(wav)
    display(Audio(out))
    files.download(out)
except urllib.error.HTTPError as e:
    print(f"HTTP {e.code}: {e.reason}")
    !tail -40 /content/chatterbox_server.log
except Exception as ex:
    print(f"Error: {ex}")
    !tail -40 /content/chatterbox_server.log


In [ ]:
# 7) Save model cache to Drive & stop
if USE_DRIVE_CACHE:
    os.makedirs(f"{DRIVE_CACHE_DIR}/hf_cache", exist_ok=True)
    print("Saving HF cache to Drive...")
    !cp -rn /content/hf_cache/. "{DRIVE_CACHE_DIR}/hf_cache/"
    print("saved")
!pkill -f "chatterbox-tts/server.py" || true
!pkill -f cloudflared || true
print("stopped")
